# Model Explainability using SHAP

This notebook generates global and local feature explanations for our final selected Ticket Type and Ticket Priority classification models.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("Warning: shap not installed. Using coefficient-based explainability fallback.")

### Load Models and Vectorizers

In [ ]:
# Ticket Type Model
tt_model = joblib.load("../models/ticket_type_model.pkl")
tt_tfidf = joblib.load("../models/ticket_type_tfidf.pkl")
tt_encoder = joblib.load("../models/ticket_type_encoder.pkl")

# Ticket Priority Model
tp_model = joblib.load("../models/ticket_priority_model.pkl")
tp_tfidf = joblib.load("../models/ticket_priority_tfidf.pkl")
tp_encoder = joblib.load("../models/ticket_priority_encoder.pkl")

# Load Data
df = pd.read_csv("../data/processed/clean_tickets.csv")
df["processed_text"] = df["processed_text"].fillna("")

### Calculate SHAP Values for Ticket Type Model

In [ ]:
# Choose a subset of 50 samples to explain
samples = df["processed_text"].iloc[:50].tolist()
X_tt = tt_tfidf.transform(samples).toarray()
tt_feature_names = tt_tfidf.get_feature_names_out()

if HAS_SHAP:
    tt_explainer = shap.Explainer(tt_model, X_tt, feature_names=tt_feature_names)
    tt_shap_values = tt_explainer(X_tt)
    print("Calculated SHAP values for Ticket Type. Shape:", tt_shap_values.shape)
else:
    print("Using coefficient-based global importance for Ticket Type.")
    if len(tt_model.coef_.shape) == 2:
        tt_global_importance = np.mean(np.abs(tt_model.coef_), axis=0)
    else:
        tt_global_importance = np.abs(tt_model.coef_)

### Ticket Type Global Summary Plot

In [ ]:
plt.figure(figsize=(10, 6))
if HAS_SHAP:
    if len(tt_shap_values.shape) == 3:
        class_idx = 0
        class_name = tt_encoder.classes_[class_idx]
        shap.summary_plot(tt_shap_values[:, :, class_idx], X_tt, feature_names=tt_feature_names, show=False)
        plt.title(f"SHAP Global Summary for Ticket Type: {class_name}")
    else:
        shap.summary_plot(tt_shap_values, X_tt, feature_names=tt_feature_names, show=False)
        plt.title("SHAP Global Summary for Ticket Type")
else:
    top_indices = np.argsort(tt_global_importance)[::-1][:20]
    plt.barh(np.array(tt_feature_names)[top_indices][::-1], tt_global_importance[top_indices][::-1], color='skyblue')
    plt.xlabel("Mean Absolute Coefficient")
    plt.title("Top 20 Feature Importances (Coefficient-based) for Ticket Type")

plt.tight_layout()
os.makedirs("../reports/figures", exist_ok=True)
plt.savefig("../reports/figures/ticket_type_shap_summary.png")
plt.close()
print("Saved Ticket Type SHAP summary plot.")

### Calculate SHAP Values for Ticket Priority Model

In [ ]:
X_tp = tp_tfidf.transform(samples).toarray()
tp_feature_names = tp_tfidf.get_feature_names_out()

if HAS_SHAP:
    tp_explainer = shap.Explainer(tp_model, X_tp, feature_names=tp_feature_names)
    tp_shap_values = tp_explainer(X_tp)
    print("Calculated SHAP values for Ticket Priority. Shape:", tp_shap_values.shape)
else:
    print("Using coefficient-based global importance for Ticket Priority.")
    if len(tp_model.coef_.shape) == 2:
        tp_global_importance = np.mean(np.abs(tp_model.coef_), axis=0)
    else:
        tp_global_importance = np.abs(tp_model.coef_)

### Ticket Priority Global Summary Plot

In [ ]:
plt.figure(figsize=(10, 6))
if HAS_SHAP:
    if len(tp_shap_values.shape) == 3:
        class_idx = 0
        class_name = tp_encoder.classes_[class_idx]
        shap.summary_plot(tp_shap_values[:, :, class_idx], X_tp, feature_names=tp_feature_names, show=False)
        plt.title(f"SHAP Global Summary for Ticket Priority: {class_name}")
    else:
        shap.summary_plot(tp_shap_values, X_tp, feature_names=tp_feature_names, show=False)
        plt.title("SHAP Global Summary for Ticket Priority")
else:
    top_indices = np.argsort(tp_global_importance)[::-1][:20]
    plt.barh(np.array(tp_feature_names)[top_indices][::-1], tp_global_importance[top_indices][::-1], color='salmon')
    plt.xlabel("Mean Absolute Coefficient")
    plt.title("Top 20 Feature Importances (Coefficient-based) for Ticket Priority")

plt.tight_layout()
plt.savefig("../reports/figures/ticket_priority_shap_summary.png")
plt.close()
print("Saved Ticket Priority SHAP summary plot.")

### Local Explanation Example

In [ ]:
sample_idx = 0
print("Sample Ticket Text:", samples[sample_idx])
pred_type_val = tt_model.predict(X_tt[sample_idx:sample_idx+1])[0]
pred_priority_val = tp_model.predict(X_tp[sample_idx:sample_idx+1])[0]

# Check if the prediction is already a string (like for Logistic Regression) or needs decoding
if isinstance(pred_type_val, str):
    predicted_type = pred_type_val
    # Get index of predicted class in classes_
    pred_type_idx = list(tt_model.classes_).index(pred_type_val)
else:
    predicted_type = tt_encoder.inverse_transform([pred_type_val])[0]
    pred_type_idx = pred_type_val

if isinstance(pred_priority_val, str):
    predicted_priority = pred_priority_val
    pred_priority_idx = list(tp_model.classes_).index(pred_priority_val)
else:
    predicted_priority = tp_encoder.inverse_transform([pred_priority_val])[0]
    pred_priority_idx = pred_priority_val

print("Predicted Type:", predicted_type)
print("Predicted Priority:", predicted_priority)

# Local explainability calculation
if HAS_SHAP:
    if len(tt_shap_values.shape) == 3:
        local_shap = tt_shap_values[sample_idx, :, pred_type_idx].values
    else:
        local_shap = tt_shap_values[sample_idx].values
else:
    coefs = tt_model.coef_[pred_type_idx] if len(tt_model.coef_.shape) == 2 else tt_model.coef_
    local_shap = coefs * X_tt[sample_idx]

top_indices = np.argsort(np.abs(local_shap))[::-1][:5]
print("\nTop 5 contributing features for Ticket Type prediction:")
for idx in top_indices:
    print(f"Feature: '{tt_feature_names[idx]}' | Weight/SHAP: {local_shap[idx]:.4f}")